# Smart Used Car Price Prediction

## 1. Objective

This Phase 1 notebook profiles the original raw dataset only. It deliberately does **not** clean, transform, encode, impute, or overwrite the source data. All findings are descriptive flags for later investigation.

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
RANDOM_STATE = 42
PROJECT_ROOT = Path("C:/Projects/smart-used-car-price-prediction")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indian_used_cars_raw_57480.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")


## 2. Dataset Configuration

The path is resolved through `pathlib`, and the source file is checked before it is read.

In [4]:
print(f"Dataset path: {DATA_PATH}")
print(f"Dataset exists: {DATA_PATH.exists()}")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Raw dataset was not found: {DATA_PATH}")


Dataset path: C:\Projects\smart-used-car-price-prediction\data\raw\indian_used_cars_raw_57480.csv
Dataset exists: True


## 3. Dataset Loading

In [6]:
# Keep the raw values and inferred dtypes exactly as supplied by pandas; do not alter df.
df = pd.read_csv(DATA_PATH)
print(f"Loaded rows: {df.shape[0]:,}")
print(f"Loaded columns: {df.shape[1]}")


Loaded rows: 57,480
Loaded columns: 11


## 4. Dataset Overview

The random sample is deterministic for reproducible inspection.

In [8]:
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head().to_string())
print("\nLast 5 rows:")
print(df.tail().to_string())
print("\nDeterministic random sample:")
print(df.sample(n=min(5, len(df)), random_state=RANDOM_STATE).to_string())
print("\nColumn names:")
print(df.columns.tolist())


Shape: (57480, 11)

First 5 rows:
   Unnamed: 0    price  manufacturing_year           model km_driven fuel_type transmission_type    brand         city bodytype  number_of_owners
0           0   840000                2024       Kia Sonet    10,000    Petrol            Manual      Kia       Rajkot      SUV                 1
1           1   998000                2023      Kia Seltos    10,107    Petrol            Manual      Kia   Coimbatore      SUV                 1
2           2   889000                2025  Citroen Basalt     5,000    Petrol            Manual  Citroen      Kolkata      SUV                 1
3           3   550000                2022   Renault Kiger    14,464    Petrol            Manual  Renault         Durg      SUV                 1
4           4  1350000                2024      Kia Carens    10,000    Diesel            Manual      Kia  Ranga reddy      MUV                 1

Last 5 rows:
       Unnamed: 0    price  manufacturing_year                model km_drive

## 5. Data Types

In [10]:
df.info()
print("\nData type of every column:")
print(df.dtypes.to_string())

numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
datetime_like_candidates = [
    col for col in df.columns
    if re.search(r"date|time|year", col, flags=re.IGNORECASE)
]
print("\nNumerical columns:", numerical_columns)
print("Categorical/object columns:", categorical_columns)
print("Datetime-like column candidates (name-based only):", datetime_like_candidates)


<class 'pandas.DataFrame'>
RangeIndex: 57480 entries, 0 to 57479
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Unnamed: 0          57480 non-null  int64
 1   price               57480 non-null  int64
 2   manufacturing_year  57480 non-null  int64
 3   model               57480 non-null  str  
 4   km_driven           57480 non-null  str  
 5   fuel_type           57452 non-null  str  
 6   transmission_type   57480 non-null  str  
 7   brand               57480 non-null  str  
 8   city                57480 non-null  str  
 9   bodytype            57480 non-null  str  
 10  number_of_owners    57480 non-null  int64
dtypes: int64(4), str(7)
memory usage: 4.8 MB

Data type of every column:
Unnamed: 0            int64
price                 int64
manufacturing_year    int64
model                   str
km_driven               str
fuel_type               str
transmission_type       str
brand                

## 6. Missing Value Analysis

In [12]:
missing_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).to_numpy(),
    "Missing Count": df.isna().sum().to_numpy(),
    "Missing Percentage": (df.isna().mean().mul(100).round(2)).to_numpy(),
    "Non-Missing Count": df.notna().sum().to_numpy(),
}).sort_values(["Missing Count", "Column"], ascending=[False, True]).reset_index(drop=True)
print(missing_report.to_string(index=False))
missing_report.to_csv(OUTPUT_DIR / "missing_value_report.csv", index=False)


            Column Data Type  Missing Count  Missing Percentage  Non-Missing Count
         fuel_type       str             28                0.05              57452
        Unnamed: 0     int64              0                0.00              57480
          bodytype       str              0                0.00              57480
             brand       str              0                0.00              57480
              city       str              0                0.00              57480
         km_driven       str              0                0.00              57480
manufacturing_year     int64              0                0.00              57480
             model       str              0                0.00              57480
  number_of_owners     int64              0                0.00              57480
             price     int64              0                0.00              57480
 transmission_type       str              0                0.00              57480


## 7. Duplicate Analysis

In [14]:
duplicate_mask = df.duplicated(keep=False)
duplicate_count = int(df.duplicated().sum())
duplicate_percentage = round(duplicate_count / len(df) * 100, 4) if len(df) else 0.0
print(f"Total duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage}%")
if duplicate_mask.any():
    print("\nSample duplicate records (inspection only):")
    print(df.loc[duplicate_mask].head(10).to_string())


Total duplicate rows: 0
Duplicate percentage: 0.0%


## 8. Numerical Feature Analysis

In [16]:
if numerical_columns:
    numerical_summary = df[numerical_columns].describe(percentiles=[0.25, 0.5, 0.75]).T
    numerical_summary = numerical_summary.rename(columns={"50%": "median"})
    numerical_summary["unique_values"] = df[numerical_columns].nunique(dropna=True)
    numerical_summary = numerical_summary[["count", "mean", "std", "min", "25%", "median", "75%", "max", "unique_values"]]
    print(numerical_summary.to_string())
else:
    numerical_summary = pd.DataFrame()
    print("No numerical columns were inferred.")
numerical_summary.to_csv(OUTPUT_DIR / "numerical_summary.csv")


                      count           mean           std      min        25%    median        75%          max  unique_values
Unnamed: 0          57480.0   28739.500000  1.659319e+04      0.0   14369.75   28739.5   43109.25      57479.0          57480
price               57480.0  996082.543650  1.832278e+06  15000.0  340000.00  550000.0  988916.00  200000000.0           3372
manufacturing_year  57480.0    2017.886517  4.383471e+00   1989.0    2015.00    2018.0    2021.00       2025.0             35
number_of_owners    57480.0       1.468615  8.535010e-01      1.0       1.00       1.0       2.00          6.0              6


## 9. Categorical Feature Analysis

Frequency tables are shown without standardizing labels, so spelling and casing patterns remain visible.

In [18]:
categorical_summary_rows = []
for col in categorical_columns:
    frequencies = df[col].value_counts(dropna=False)
    categorical_summary_rows.append({
        "Column": col,
        "Unique Count": int(df[col].nunique(dropna=True)),
        "Missing Count": int(df[col].isna().sum()),
        "Top Categories": " | ".join(f"{str(value)} ({count})" for value, count in frequencies.head(15).items()),
    })
    print(f"\n{col} — unique={df[col].nunique(dropna=True):,}, missing={df[col].isna().sum():,}")
    print(frequencies.head(20).to_string())
categorical_summary = pd.DataFrame(categorical_summary_rows)
categorical_summary.to_csv(OUTPUT_DIR / "categorical_summary.csv", index=False)



model — unique=1,200, missing=0
model
Maruti Swift          1455
Maruti Wagon R        1393
Hyundai i20           1279
Maruti Alto 800       1175
Maruti Swift Dzire    1162
Hyundai Creta          994
Maruti Baleno          981
Honda City             808
Hyundai Grand i10      773
Kia Seltos             695
Hyundai i10            658
Tata Nexon             637
Tata Tiago             624
Renault KWID           614
Maruti Alto K10        612
Maruti Ertiga          611
Maruti Celerio         559
Honda Amaze            556
Hyundai Venue          555
Hyundai Verna          535

km_driven — unique=10,502, missing=0
km_driven
1,00,000    1286
1,20,000    1271
80,000      1245
60,000      1215
70,000      1180
50,000      1075
90,000       968
40,000       924
1,50,000     913
30,000       886
1,60,000     780
20,000       766
10,000       578
100000       409
70000        326
65000        316
50000        313
65,000       290
80000        286
60000        280

fuel_type — unique=9, missing=28

## 10. Cardinality Analysis

In [20]:
cardinality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).to_numpy(),
    "Unique Count": df.nunique(dropna=True).to_numpy(),
    "Unique Percentage": (df.nunique(dropna=True).div(len(df)).mul(100).round(2)).to_numpy(),
}).sort_values(["Unique Percentage", "Column"], ascending=[False, True]).reset_index(drop=True)
print(cardinality_report.to_string(index=False))
cardinality_report.to_csv(OUTPUT_DIR / "dataset_profile.csv", index=False)


            Column Data Type  Unique Count  Unique Percentage
        Unnamed: 0     int64         57480             100.00
         km_driven       str         10502              18.27
             price     int64          3372               5.87
              city       str          1468               2.55
             model       str          1200               2.09
             brand       str            47               0.08
manufacturing_year     int64            35               0.06
          bodytype       str            18               0.03
         fuel_type       str             9               0.02
  number_of_owners     int64             6               0.01
 transmission_type       str             8               0.01


## 11. Target Candidate Investigation

Candidates are shortlisted by column-name keywords and then described; this does not select a final target.

In [22]:
target_pattern = re.compile(r"price|selling[_ ]?price|resale[_ ]?price|amount|cost", re.IGNORECASE)
target_candidates = [col for col in df.columns if target_pattern.search(str(col))]
candidate_rows = []
for col in target_candidates:
    series = df[col]
    is_numeric = pd.api.types.is_numeric_dtype(series)
    candidate_rows.append({
        "Column": col,
        "Data Type": str(series.dtype),
        "Unique Values": int(series.nunique(dropna=True)),
        "Missing Values": int(series.isna().sum()),
        "Numerical Summary": (series.describe().to_dict() if is_numeric else "Not numerical"),
        "Appears price-related": bool(target_pattern.search(str(col))),
        "Assessment": "Name matches a price-related keyword; values require business review before target selection.",
    })
target_candidate_report = pd.DataFrame(candidate_rows)
if target_candidate_report.empty:
    print("No name-based target candidates were found.")
else:
    print(target_candidate_report.to_string(index=False))
target_candidate_report.to_csv(OUTPUT_DIR / "target_candidate_report.csv", index=False)


Column Data Type  Unique Values  Missing Values                                                                                                                                               Numerical Summary  Appears price-related                                                                                    Assessment
 price     int64           3372               0 {'count': 57480.0, 'mean': 996082.5436499652, 'std': 1832278.3332336568, 'min': 15000.0, '25%': 340000.0, '50%': 550000.0, '75%': 988916.0, 'max': 200000000.0}                   True Name matches a price-related keyword; values require business review before target selection.


## 12. Initial Visual Analysis

A compact set of plots is used to inspect distributions. No outliers are removed or altered.

In [24]:
# Limit numerical inspection to a small, readable set.
plot_columns = numerical_columns[:min(4, len(numerical_columns))]
if plot_columns:
    fig, axes = plt.subplots(len(plot_columns), 2, figsize=(14, 4 * len(plot_columns)))
    axes = np.atleast_2d(axes)
    for row, col in enumerate(plot_columns):
        sns.histplot(data=df, x=col, bins=40, ax=axes[row, 0], color="#2a6fbb")
        axes[row, 0].set_title(f"Distribution: {col}")
        sns.boxplot(data=df, x=col, ax=axes[row, 1], color="#8ecae6")
        axes[row, 1].set_title(f"Boxplot: {col}")
    fig.tight_layout()
    plt.close(fig)

low_cardinality = [col for col in categorical_columns if df[col].nunique(dropna=True) <= 20][:4]
if low_cardinality:
    fig, axes = plt.subplots(len(low_cardinality), 1, figsize=(12, 4 * len(low_cardinality)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, low_cardinality):
        counts = df[col].value_counts(dropna=False).head(20)
        sns.barplot(x=counts.values, y=counts.index.astype(str), ax=ax, color="#ffb703")
        ax.set_title(f"Category frequency: {col}")
        ax.set_xlabel("Rows")
        ax.set_ylabel(col)
    fig.tight_layout()
    plt.close(fig)


## 13. Data Quality Flags

These are preliminary signals, not validity decisions or cleaning rules.

In [26]:
flags = []
for _, row in missing_report.loc[missing_report["Missing Count"] > 0].iterrows():
    flags.append({"Concern": "Missing values", "Column(s)": row["Column"], "Evidence": f"{row['Missing Count']:,} missing ({row['Missing Percentage']:.2f}%).", "Requires investigation": "Yes"})
if duplicate_count:
    flags.append({"Concern": "Duplicate records", "Column(s)": "All columns", "Evidence": f"{duplicate_count:,} duplicate rows ({duplicate_percentage}%).", "Requires investigation": "Yes"})
for col in numerical_columns:
    negative_count = int((df[col] < 0).sum())
    zero_count = int((df[col] == 0).sum())
    if negative_count:
        flags.append({"Concern": "Negative numerical values", "Column(s)": col, "Evidence": f"{negative_count:,} negative values observed.", "Requires investigation": "Yes"})
    if zero_count:
        flags.append({"Concern": "Zero numerical values", "Column(s)": col, "Evidence": f"{zero_count:,} zero values observed; context is needed to judge them.", "Requires investigation": "Yes"})
for col in datetime_like_candidates:
    if pd.api.types.is_numeric_dtype(df[col]):
        out_of_range = int(((df[col] < 1900) | (df[col] > pd.Timestamp.today().year + 1)).sum())
        if out_of_range:
            flags.append({"Concern": "Potentially unusual year", "Column(s)": col, "Evidence": f"{out_of_range:,} values fall outside 1900 to next calendar year.", "Requires investigation": "Yes"})
for _, row in cardinality_report.loc[cardinality_report["Unique Percentage"] > 50].iterrows():
    flags.append({"Concern": "High cardinality", "Column(s)": row["Column"], "Evidence": f"{row['Unique Count']:,} unique values ({row['Unique Percentage']:.2f}%).", "Requires investigation": "Yes"})
quality_flags = pd.DataFrame(flags, columns=["Concern", "Column(s)", "Evidence", "Requires investigation"])
print(quality_flags.to_string(index=False) if not quality_flags.empty else "No preliminary flags were triggered by the defined descriptive checks.")
quality_flags.to_csv(OUTPUT_DIR / "data_quality_flags.csv", index=False)


              Concern  Column(s)                                                 Evidence Requires investigation
       Missing values  fuel_type                                      28 missing (0.05%).                    Yes
Zero numerical values Unnamed: 0 1 zero values observed; context is needed to judge them.                    Yes
     High cardinality Unnamed: 0                          57,480 unique values (100.00%).                    Yes


## 14. Preliminary Data Dictionary

In [28]:
def infer_role(column: str) -> str:
    name = str(column).lower()
    if name.startswith("unnamed") or name in {"id", "index"}:
        return "Identifier"
    if column in target_candidates:
        return "Target Candidate"
    return "Feature"

dictionary_rows = []
for col in df.columns:
    concerns = quality_flags.loc[quality_flags["Column(s)"].eq(col), "Concern"].tolist()
    if col in cardinality_report.loc[cardinality_report["Unique Percentage"] > 50, "Column"].tolist():
        concerns.append("High cardinality")
    dictionary_rows.append({
        "Column Name": col,
        "Data Type": str(df[col].dtype),
        "Unique Count": int(df[col].nunique(dropna=True)),
        "Missing Count": int(df[col].isna().sum()),
        "Likely Role": infer_role(col),
        "Initial Interpretation": "Observed column; meaning has not been confirmed from external documentation.",
        "Potential Data Quality Concern": "; ".join(dict.fromkeys(concerns)) if concerns else "None flagged by initial checks",
    })
data_dictionary = pd.DataFrame(dictionary_rows)
print(data_dictionary.to_string(index=False))
data_dictionary.to_csv(OUTPUT_DIR / "data_dictionary_initial.csv", index=False)


       Column Name Data Type  Unique Count  Missing Count      Likely Role                                                       Initial Interpretation          Potential Data Quality Concern
        Unnamed: 0     int64         57480              0       Identifier Observed column; meaning has not been confirmed from external documentation. Zero numerical values; High cardinality
             price     int64          3372              0 Target Candidate Observed column; meaning has not been confirmed from external documentation.          None flagged by initial checks
manufacturing_year     int64            35              0          Feature Observed column; meaning has not been confirmed from external documentation.          None flagged by initial checks
             model       str          1200              0          Feature Observed column; meaning has not been confirmed from external documentation.          None flagged by initial checks
         km_driven       str         105

## 15. Profiling Summary

This summary is computed from the loaded dataset and deliberately avoids recommending data changes.

In [30]:
missing_columns = missing_report.loc[missing_report["Missing Count"] > 0, "Column"].tolist()
high_cardinality_columns = cardinality_report.loc[cardinality_report["Unique Percentage"] > 50, "Column"].tolist()
print(f"- Dataset contains {len(df):,} rows and {df.shape[1]} columns.")
print(f"- {len(numerical_columns)} columns were inferred as numerical and {len(categorical_columns)} as categorical/object.")
print(f"- Missing values were observed in: {missing_columns or 'none'}.")
print(f"- Duplicate rows observed: {duplicate_count:,}.")
print(f"- Name-based potential target candidates: {target_candidates or 'none'}.")
print(f"- High-cardinality columns: {high_cardinality_columns or 'none under the >50% unique rule'}.")
print("- Any preliminary flags above require investigation; no raw values were modified.")


- Dataset contains 57,480 rows and 11 columns.
- 4 columns were inferred as numerical and 7 as categorical/object.
- Missing values were observed in: ['fuel_type'].
- Duplicate rows observed: 0.
- Name-based potential target candidates: ['price'].
- High-cardinality columns: ['Unnamed: 0'].
- Any preliminary flags above require investigation; no raw values were modified.


## 16. Target and Suspicious Value Investigation

This evidence-only investigation uses the original `df` and temporary inspection series. It does not alter rows, values, names, or the raw source file.


### 16.1 Price Investigation


In [33]:
price_series = df["price"]
price_percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
price_statistics = pd.Series({
    "dtype": str(price_series.dtype), "minimum": price_series.min(), "maximum": price_series.max(),
    "mean": price_series.mean(), "median": price_series.median(), "standard_deviation": price_series.std(),
    "unique_values": price_series.nunique(dropna=True), "zero_values": int((price_series == 0).sum()),
    "negative_values": int((price_series < 0).sum()),
})
for percentile, value in price_series.quantile(price_percentiles).items():
    price_statistics[f"percentile_{percentile * 100:g}%"] = value
print("Price statistics:\n", price_statistics.to_string())
print("\n20 lowest price records (all original columns):\n", df.loc[price_series.nsmallest(20).index].to_string())
print("\n20 highest price records (all original columns):\n", df.loc[price_series.nlargest(20).index].to_string())


Price statistics:
 dtype                          int64
minimum                        15000
maximum                    200000000
mean                    996082.54365
median                      550000.0
standard_deviation    1832278.333234
unique_values                   3372
zero_values                        0
negative_values                    0
percentile_1%                70000.0
percentile_5%               140000.0
percentile_10%              200000.0
percentile_25%              340000.0
percentile_50%              550000.0
percentile_75%              988916.0
percentile_90%             1845000.0
percentile_95%             3450000.0
percentile_99%             8300000.0
percentile_99.5%          10000000.0
percentile_99.9%          15500000.0

20 lowest price records (all original columns):
        Unnamed: 0  price  manufacturing_year                model km_driven fuel_type transmission_type       brand                city   bodytype  number_of_owners
16028       16028  15000  

### 16.2 Kilometres Driven Investigation

`km_driven` remains unchanged. The temporary series removes display commas solely for inspection.


In [35]:
km_driven_numeric = pd.to_numeric(df["km_driven"].astype("string").str.replace(",", "", regex=False), errors="coerce")
km_parse_failures = int(km_driven_numeric.isna().sum() - df["km_driven"].isna().sum())
km_percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
km_statistics = pd.Series({
    "original_dtype": str(df["km_driven"].dtype), "temporary_numeric_parse_failures": km_parse_failures,
    "minimum": km_driven_numeric.min(), "maximum": km_driven_numeric.max(), "mean": km_driven_numeric.mean(),
    "median": km_driven_numeric.median(), "standard_deviation": km_driven_numeric.std(),
    "zero_values": int((km_driven_numeric == 0).sum()), "negative_values": int((km_driven_numeric < 0).sum()),
})
for percentile, value in km_driven_numeric.quantile(km_percentiles).items():
    km_statistics[f"percentile_{percentile * 100:g}%"] = value
print("km_driven statistics from temporary numeric inspection series:\n", km_statistics.to_string())
print("\n20 highest km_driven records (all original columns):\n", df.loc[km_driven_numeric.nlargest(20).index].to_string())


km_driven statistics from temporary numeric inspection series:
 original_dtype                               str
temporary_numeric_parse_failures               0
minimum                                        2
maximum                                  3750000
mean                                72688.271642
median                                   63000.0
standard_deviation                  73624.080291
zero_values                                    0
negative_values                                0
percentile_1%                             3000.0
percentile_5%                            11000.0
percentile_25%                           37100.0
percentile_50%                           63000.0
percentile_75%                           93000.0
percentile_90%                          131015.2
percentile_95%                          160000.0
percentile_99%                          230000.0
percentile_99.5%                        261210.0
percentile_99.9%                       1042605.0

20 h

### 16.3 Manufacturing Year Investigation


In [37]:
year_series = df["manufacturing_year"]
year_frequencies = year_series.value_counts(dropna=False).sort_index()
years_requiring_review = sorted(year_series[year_series <= 2000].dropna().unique().tolist())
print(f"Minimum manufacturing year: {year_series.min()}")
print(f"Maximum manufacturing year: {year_series.max()}")
print("Unique years:", sorted(year_series.dropna().unique().tolist()))
print("\nFrequency by year:\n", year_frequencies.to_string())
print("\nObserved old-year tail selected for investigation (<= 2000):", years_requiring_review)
if years_requiring_review:
    print(df.loc[year_series.isin(years_requiring_review)].sort_values("manufacturing_year").to_string())


Minimum manufacturing year: 1989
Maximum manufacturing year: 2025
Unique years: [1989, 1990, 1992, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Frequency by year:
 manufacturing_year
1989       1
1990       1
1992       1
1994       1
1995       2
1996       7
1997       3
1998      11
1999      14
2000      28
2001      37
2002      26
2003      49
2004      80
2005     168
2006     221
2007     331
2008     414
2009     629
2010    1196
2011    1711
2012    2281
2013    2493
2014    2970
2015    3422
2016    4110
2017    4597
2018    5083
2019    4473
2020    3968
2021    4847
2022    5412
2023    4563
2024    3049
2025    1281

Observed old-year tail selected for investigation (<= 2000): [1989, 1990, 1992, 1994, 1995, 1996, 1997, 1998, 1999, 2000]
       Unnamed: 0    price  manufacturing_year                  model km_driven fuel_type t

### 16.4 Missing `fuel_type` Values


In [39]:
missing_fuel_rows = df.loc[df["fuel_type"].isna()]
print(f"Rows with missing fuel_type: {len(missing_fuel_rows)}")
print(missing_fuel_rows.to_string())


Rows with missing fuel_type: 28
       Unnamed: 0    price  manufacturing_year                    model km_driven fuel_type transmission_type          brand             city   bodytype  number_of_owners
35718       35718  4800000                2025                      EV6      4000       NaN         Automatic            Kia             Agra        SUV                 1
36281       36281  1400000                2016                   Cresta     65000       NaN            Manual         Toyota           Jhansi      Sedan                 1
38206       38206    90000                2008         Alto [2010-2013]     56896       NaN            Manual  Maruti Suzuki            Banda  Hatchback                 2
39555       39555  1300000                2017                   Cresta    200000       NaN            Manual         Toyota        Alappuzha      Sedan                 3
40680       40680    45000                1998                     Omni    102500       NaN            Manual  Ma

### 16.5 `Unnamed: 0` Investigation


In [41]:
unnamed_series = df["Unnamed: 0"]
expected_sequence = pd.Series(range(len(df)), dtype=unnamed_series.dtype)
unnamed_investigation = pd.Series({
    "unique_values": unnamed_series.nunique(dropna=True),
    "is_sequential_from_zero": bool(unnamed_series.reset_index(drop=True).equals(expected_sequence)),
    "duplicates_dataframe_index": bool(unnamed_series.reset_index(drop=True).equals(pd.Series(df.index, dtype=unnamed_series.dtype))),
})
print("First 20 values:", unnamed_series.head(20).tolist())
print("Last 20 values:", unnamed_series.tail(20).tolist())
print(unnamed_investigation.to_string())


First 20 values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Last 20 values: [57460, 57461, 57462, 57463, 57464, 57465, 57466, 57467, 57468, 57469, 57470, 57471, 57472, 57473, 57474, 57475, 57476, 57477, 57478, 57479]
unique_values                 57480
is_sequential_from_zero        True
duplicates_dataframe_index     True


### 16.6 Potential Data Quality Flags

These flags do not prescribe cleaning or deletion.


In [43]:
focused_flags = pd.DataFrame([
    {"Column": "price", "Issue": "Very wide upper-tail distribution", "Observed Evidence": f"Minimum={price_series.min():,.0f}; 99.9th percentile={price_series.quantile(0.999):,.0f}; maximum={price_series.max():,.0f}.", "Rows Affected": int((price_series > price_series.quantile(0.999)).sum()), "Severity": "Requires investigation", "Action Required": "Needs investigation"},
    {"Column": "km_driven", "Issue": "Numeric-looking values stored as text", "Observed Evidence": f"dtype={df['km_driven'].dtype}; temporary comma-removal parse failures={km_parse_failures}.", "Rows Affected": int(df["km_driven"].notna().sum()), "Severity": "Requires investigation", "Action Required": "Needs investigation"},
    {"Column": "fuel_type", "Issue": "Missing values", "Observed Evidence": f"Missing count={len(missing_fuel_rows):,} ({len(missing_fuel_rows) / len(df) * 100:.2f}%).", "Rows Affected": len(missing_fuel_rows), "Severity": "Requires investigation", "Action Required": "Needs investigation"},
    {"Column": "Unnamed: 0", "Issue": "Index-like duplicate of DataFrame index", "Observed Evidence": f"Unique={unnamed_series.nunique():,}; sequential from zero={unnamed_investigation['is_sequential_from_zero']}; duplicates DataFrame index={unnamed_investigation['duplicates_dataframe_index']}.", "Rows Affected": len(df), "Severity": "Requires investigation", "Action Required": "Needs investigation"},
    {"Column": "manufacturing_year", "Issue": "Oldest observed year tail", "Observed Evidence": f"Range={year_series.min()}–{year_series.max()}; {int((year_series <= 2000).sum()):,} rows are from years <= 2000.", "Rows Affected": int((year_series <= 2000).sum()), "Severity": "Requires investigation", "Action Required": "Needs investigation"},
])
print(focused_flags.to_string(index=False))
focused_flags.to_csv(OUTPUT_DIR / "focused_data_quality_flags.csv", index=False)


            Column                                   Issue                                                          Observed Evidence  Rows Affected               Severity     Action Required
             price       Very wide upper-tail distribution         Minimum=15,000; 99.9th percentile=15,500,000; maximum=200,000,000.             56 Requires investigation Needs investigation
         km_driven   Numeric-looking values stored as text                       dtype=str; temporary comma-removal parse failures=0.          57480 Requires investigation Needs investigation
         fuel_type                          Missing values                                                  Missing count=28 (0.05%).             28 Requires investigation Needs investigation
        Unnamed: 0 Index-like duplicate of DataFrame index Unique=57,480; sequential from zero=True; duplicates DataFrame index=True.          57480 Requires investigation Needs investigation
manufacturing_year               Oldest 

### 16.7 Target Candidate Conclusion


In [45]:
print("Observed conclusion: `price` is a reasonable target candidate for later review because its name is price-related, "
      f"it is numeric ({price_series.dtype}), has {price_series.notna().sum():,} non-missing values, and contains {price_series.nunique():,} distinct values. "
      "This does not validate every value; the observed upper tail requires investigation.")


Observed conclusion: `price` is a reasonable target candidate for later review because its name is price-related, it is numeric (int64), has 57,480 non-missing values, and contains 3,372 distinct values. This does not validate every value; the observed upper tail requires investigation.


### 16.8 Updated Final Profiling Summary


In [47]:
print(f"- Dataset remains {len(df):,} rows by {df.shape[1]} columns; this notebook did not modify it.")
print(f"- `price`: dtype={price_series.dtype}, min={price_series.min():,.0f}, median={price_series.median():,.0f}, max={price_series.max():,.0f}; zero/negative={int((price_series == 0).sum())}/{int((price_series < 0).sum())}.")
print(f"- `km_driven`: original dtype={df['km_driven'].dtype}; parse failures={km_parse_failures}; min={km_driven_numeric.min():,.0f}, median={km_driven_numeric.median():,.0f}, max={km_driven_numeric.max():,.0f}.")
print(f"- `manufacturing_year` spans {year_series.min()} to {year_series.max()}; {int((year_series <= 2000).sum()):,} rows are in the <=2000 review tail.")
print(f"- `fuel_type` has {len(missing_fuel_rows):,} missing rows, displayed without filling.")
print(f"- `Unnamed: 0` sequentially duplicates the DataFrame index: {unnamed_investigation['duplicates_dataframe_index']}.")
print("- All data-quality issues require investigation only; no cleaning, deletion, or modelling was performed.")


- Dataset remains 57,480 rows by 11 columns; this notebook did not modify it.
- `price`: dtype=int64, min=15,000, median=550,000, max=200,000,000; zero/negative=0/0.
- `km_driven`: original dtype=str; parse failures=0; min=2, median=63,000, max=3,750,000.
- `manufacturing_year` spans 1989 to 2025; 69 rows are in the <=2000 review tail.
- `fuel_type` has 28 missing rows, displayed without filling.
- `Unnamed: 0` sequentially duplicates the DataFrame index: True.
- All data-quality issues require investigation only; no cleaning, deletion, or modelling was performed.
